In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [12]:
class evaluateSchema(BaseModel):
    approval: Literal['approved', 'needs_improvement']
    feedback: str

model = ChatOpenAI(model="gpt-4o-mini")
evaluate_model = ChatOpenAI(model="gpt-4o")
structured_model = evaluate_model.with_structured_output(evaluateSchema)
optimize_model = ChatOpenAI(model="gpt-4o-mini")

In [13]:
class TweetState(TypedDict):
    tweet: str
    title: str
    evaluation: Literal['approved', 'needs_improvement']
    feedback: str
    iteration: int
    max_iterations: int


In [14]:
def generate_tweet(state: TweetState) :
    prompt = f"You are a witty tweet writer. Write a tweet for the following title: {state['title']}"
    response = model.invoke(prompt).content
    return {'tweet': response, 'iteration': 1}

def evaluate_tweet(state: TweetState) :
    prompt = f"You are a social media manager. Evaluate the following tweet: {state['tweet']}. Tell if its approved or needs improvement. Provide feedback."
    response = structured_model.invoke(prompt)
    return {'evaluation': response.approval, 'feedback': response.feedback}

def optimize_tweet(state: TweetState) :
    prompt = f"you are a experienced witty tweet writer. You are given a tweet and a feedback. Optimize the tweet to improve engagement make it more engaging and funny. Tweet: {state['tweet']}. Feedback: {state['feedback']}"
    response = optimize_model.invoke(prompt).content
    return {'tweet': response, 'iteration': state['iteration'] + 1}

def check_approval(state: TweetState) :
    if state['evaluation'] == 'approved' :
        return 'approved'
    else :
        return 'needs_improvement'


In [15]:
graph = StateGraph(TweetState)

graph.add_node('evaluate_tweet', evaluate_tweet)
graph.add_node('generate_tweet', generate_tweet)
graph.add_node('optimize_tweet', optimize_tweet)

graph.add_edge(START, 'generate_tweet')
graph.add_edge('generate_tweet', 'evaluate_tweet')
graph.add_conditional_edges('evaluate_tweet', check_approval, {'approved': END, 'needs_improvement': 'optimize_tweet'})
graph.add_edge('optimize_tweet', 'evaluate_tweet')

workflow = graph.compile()



In [16]:
initial_state = {"title": "The best ice cream in the world", "max_iterations": 5}

workflow.invoke(initial_state)


{'tweet': '🌍🍦 If "best ice cream in the world" were a reality show, it would need a voting system, a panel of judges, and at least one dramatic backstory involving a runaway cow. 🐄💔 But let\'s be real: whoever\'s scooping has already won my heart! #IceCreamDreams #ScoopGoals',
 'title': 'The best ice cream in the world',
 'evaluation': 'approved',
 'feedback': 'This tweet is engaging and creative, capturing the playful and sometimes whimsical nature of social media communication. Here are some specific points that contribute to its success:\n\n1. **Use of Emojis**: The clever use of emojis adds visual interest and helps convey the light-hearted tone of the tweet. The choice of globe, ice cream, cow, and broken heart emojis perfectly complements the narrative.\n\n2. **Imagery and Humor**: By painting the scene of a reality show involving dramatic backstories and a runaway cow, the tweet captures attention through vivid imagery and humor.\n\n3. **Engagement through Relatability**: The tw